### TASK A

In [2]:
# ============================================================================
# IMPORTS AND CONFIGURATION
# ============================================================================

import os
import json
import random
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Any
from dataclasses import dataclass
from collections import defaultdict, Counter
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    XLMRobertaTokenizer, XLMRobertaForTokenClassification,
    XLMRobertaForSequenceClassification, AutoTokenizer,
    Trainer, TrainingArguments, pipeline, set_seed
)

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_recall_fscore_support
)
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    XLMRobertaTokenizerFast
)
from datasets import load_dataset
from datasets import concatenate_datasets
import torch

import numpy as np
# Set random seeds for reproducibility
set_seed(42)
torch.manual_seed(42)
np.random.seed(42)

DATASET : i choosed finance dataset

In [ ]:
dataset = load_dataset("gretelai/synthetic_pii_finance_multilingual")

sample = dataset["train"][0]
sample

In [ ]:
sample["pii_spans"]

MODEL

In [3]:
# ------------------------------------------------------------------
# Encode example → token-level BIO labels
# ------------------------------------------------------------------

def encode_finance_example(example, tokenizer, label2id, max_length=128):
    text = example["generated_text"]

    #  FIX: parse JSON string
    spans = json.loads(example["pii_spans"])

    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_offsets_mapping=True)

    labels = ["O"] * len(encoding["input_ids"])

    for span in spans:
        start, end = span["start"], span["end"]
        label = span["label"].upper()

        for i, (s, e) in enumerate(encoding["offset_mapping"]):
            if s >= end or e <= start:
                continue

            prefix = "B-" if s == start else "I-"
            tag = prefix + label

            if tag in label2id:
                labels[i] = tag

    encoding["labels"] = [
        label2id[l] if l in label2id else -100
        for l in labels
    ]

    encoding.pop("offset_mapping")
    return encoding


In [4]:
# ============================================================================
# MULTILINGUAL NER MODEL (FIXED VERSION)
# ============================================================================

class MultilingualNERModel:
    """Fine-tune multilingual Transformer for Financial PII NER"""

    def __init__(self, model_name="xlm-roberta-base"):
        self.model_name = model_name
        self.tokenizer = XLMRobertaTokenizerFast.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Define label mappings
        self.label_list = [
            "O",
            "B-NAME", "I-NAME",
            "B-DATE", "I-DATE",
            "B-STREET_ADDRESS", "I-STREET_ADDRESS",
            "B-BBAN", "I-BBAN",
            "B-IBAN", "I-IBAN",
            "B-EMAIL", "I-EMAIL",
            "B-PHONE", "I-PHONE",
            "B-ORGANIZATION", "I-ORGANIZATION"
        ]
        self.label2id = {label: i for i, label in enumerate(self.label_list)}
        self.id2label = {i: label for i, label in enumerate(self.label_list)}

        # Initialize model
        self.model = XLMRobertaForTokenClassification.from_pretrained(
            model_name,
            num_labels=len(self.label_list),
            id2label=self.id2label,
            label2id=self.label2id
        ).to(self.device)

    def train(self, train_dataset, eval_dataset=None, epochs=3, batch_size=16):
        """Train the NER model"""

        training_args = TrainingArguments(
            output_dir="./ner_model",
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            logging_dir="./logs",
            logging_steps=100,
            save_strategy="epoch",
            eval_strategy="epoch" if eval_dataset is not None else "no",
            load_best_model_at_end=True if eval_dataset is not None else False,
            weight_decay=0.01,
            metric_for_best_model="eval_overall_accuracy",
            greater_is_better=True,
            save_total_limit=2,
            push_to_hub=False,
            report_to=[],  # disables W&B logging
            fp16=True if torch.cuda.is_available() else False
        )

        def compute_metrics(p):
            predictions, labels = p
            predictions = np.argmax(predictions, axis=2)

            # Remove ignored index (special tokens)
            true_predictions = [
                [self.label_list[p] for (p, l) in zip(prediction, label) if l != -100]
                for prediction, label in zip(predictions, labels)
            ]
            true_labels = [
                [self.label_list[l] for (p, l) in zip(prediction, label) if l != -100]
                for prediction, label in zip(predictions, labels)
            ]

            # Calculate metrics per entity type
            all_entity_types = [
                "NAME", "DATE", "STREET_ADDRESS",
                "BBAN", "IBAN", "EMAIL", "PHONE", "ORGANIZATION"
            ]
            results = {}

            for entity_type in all_entity_types:
                # Get predictions/labels for this entity type
                preds_flat = []
                labels_flat = []

                for pred_seq, label_seq in zip(true_predictions, true_labels):
                    for pred, label in zip(pred_seq, label_seq):
                        pred_entity = pred.split("-")[-1] if "-" in pred else "O"
                        label_entity = label.split("-")[-1] if "-" in label else "O"

                        # Binary classification for this entity
                        preds_flat.append(1 if pred_entity == entity_type else 0)
                        labels_flat.append(1 if label_entity == entity_type else 0)

                # Calculate metrics
                precision, recall, f1, _ = precision_recall_fscore_support(
                    labels_flat, preds_flat, average="binary", zero_division=0
                )

                results[f"{entity_type.lower()}_precision"] = precision
                results[f"{entity_type.lower()}_recall"] = recall
                results[f"{entity_type.lower()}_f1"] = f1

            # Overall metrics
            flat_preds = [p for seq in true_predictions for p in seq]
            flat_labels = [l for seq in true_labels for l in seq]

            results["overall_accuracy"] = accuracy_score(flat_labels, flat_preds)

            return results

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            tokenizer=self.tokenizer,
            compute_metrics=compute_metrics
        )

        trainer.train()

        return trainer

    def predict(self, texts: list[str], language: str = "en") -> list[list[dict]]:
        """Predict entities in texts using token offsets (FIXED VERSION)"""
        self.model.eval()
        all_predictions = []

        for text in texts:
            # Tokenize the raw text to get proper offset_mapping
            encoding = self.tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                padding=True,
                return_offsets_mapping=True
            ).to(self.device)

            # Extract offset_mapping for later use
            offset_mapping = encoding.pop("offset_mapping").squeeze(0)  # Remove from model inputs

            with torch.no_grad():
                # Forward pass with only model-expected inputs
                outputs = self.model(**encoding)
                predictions = torch.argmax(outputs.logits, dim=-1).squeeze(0)  # shape: (seq_len,)

            # Get offset mapping back to CPU
            offsets = offset_mapping.cpu().numpy()  # shape: (seq_len, 2)
            entities = []
            current_entity = None

            # Convert token predictions to character-level entities
            for (start, end), pred_id in zip(offsets, predictions):
                # Skip special tokens (CLS, SEP, PAD)
                if start == end:
                    # If we were tracking an entity, end it before special token
                    if current_entity:
                        entities.append(current_entity)
                        current_entity = None
                    continue

                label = self.id2label[pred_id.item()]

                if label.startswith("B-"):
                    # Start new entity
                    if current_entity:
                        entities.append(current_entity)
                    current_entity = {
                        "entity": label[2:],  # Remove "B-" prefix
                        "start_char": int(start),
                        "end_char": int(end)
                    }
                elif label.startswith("I-") and current_entity:
                    # Continue current entity (check if same entity type)
                    if current_entity["entity"] == label[2:]:
                        current_entity["end_char"] = int(end)
                    else:
                        # Different entity type, end current and start new
                        entities.append(current_entity)
                        current_entity = {
                            "entity": label[2:],
                            "start_char": int(start),
                            "end_char": int(end)
                        }
                else:  # "O" label or invalid state
                    if current_entity:
                        entities.append(current_entity)
                        current_entity = None

            # Append any last entity
            if current_entity:
                entities.append(current_entity)

            # Post-process: Merge adjacent entities of same type
            merged_entities = []
            for ent in entities:
                if (merged_entities and
                    merged_entities[-1]["entity"] == ent["entity"] and
                    merged_entities[-1]["end_char"] == ent["start_char"]):
                    # Merge with previous entity
                    merged_entities[-1]["end_char"] = ent["end_char"]
                else:
                    merged_entities.append(ent)

            # Add the original word and extra info
            token_entities = []
            for ent in merged_entities:
                # Extract text from original string
                extracted_text = text[ent["start_char"]:ent["end_char"]]
                # Calculate confidence (simple approximation)
                confidence = 0.95  # Could be enhanced with logits

                token_entities.append({
                    "entity": ent["entity"],
                    "word": extracted_text,
                    "start_char": ent["start_char"],
                    "end_char": ent["end_char"],
                    "confidence": confidence,
                    "language": language
                })

            all_predictions.append(token_entities)

        return all_predictions

    def evaluate(self, test_dataset, batch_size=16):
        """Evaluate model on test dataset"""
        self.model.eval()

        eval_args = TrainingArguments(
            output_dir="./eval",
            per_device_eval_batch_size=batch_size,
            report_to=[]
        )

        trainer = Trainer(
            model=self.model,
            args=eval_args,
            tokenizer=self.tokenizer
        )

        results = trainer.evaluate(test_dataset)
        return results

    def evaluate_cross_lingual(self, en_dataset, fr_dataset):
        """Evaluate cross-lingual transfer performance"""
        print("="*70)
        print("CROSS-LINGUAL TRANSFER EVALUATION")
        print("="*70)

        # Evaluate on English
        print("\n1. English Evaluation:")
        en_results = self.evaluate(en_dataset)

        # Evaluate on French
        print("\n2. French Evaluation:")
        fr_results = self.evaluate(fr_dataset)

        # Calculate language-specific metrics
        metrics = {
            "en": {
                "precision": en_results.get("eval_name_precision", 0),
                "recall": en_results.get("eval_name_recall", 0),
                "f1": en_results.get("eval_name_f1", 0)
            },
            "fr": {
                "precision": fr_results.get("eval_name_precision", 0),
                "recall": fr_results.get("eval_name_recall", 0),
                "f1": fr_results.get("eval_name_f1", 0)
            }
        }

        # Visualize cross-lingual performance
        self.plot_cross_lingual_metrics(metrics)

        return metrics

    def plot_cross_lingual_metrics(self, metrics):
        """Plot cross-lingual performance comparison"""
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Bar chart for F1 scores
        languages = list(metrics.keys())
        f1_scores = [metrics[lang]["f1"] for lang in languages]

        axes[0].bar(languages, f1_scores, color=['blue', 'red'])
        axes[0].set_title("F1 Scores by Language")
        axes[0].set_ylabel("F1 Score")
        axes[0].set_ylim(0, 1)

        # Precision-Recall comparison
        for i, lang in enumerate(languages):
            axes[1].scatter(metrics[lang]["precision"], metrics[lang]["recall"],
                          label=lang, s=100, alpha=0.7)

        axes[1].set_xlabel("Precision")
        axes[1].set_ylabel("Recall")
        axes[1].set_title("Precision-Recall Tradeoff")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig("cross_lingual_metrics.png", dpi=150)
        plt.show()

    def save_model(self, path="./multilingual_ner_model"):
        """Save the trained model"""
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        print(f"Model saved to {path}")

    def load_model(self, path="./multilingual_ner_model"):
        """Load a trained model"""
        self.model = XLMRobertaForTokenClassification.from_pretrained(path).to(self.device)
        self.tokenizer = XLMRobertaTokenizerFast.from_pretrained(path)
        print(f"Model loaded from {path}")

In [11]:
def main():
    print("\n" + "=" * 70)
    print("TASK A: MULTILINGUAL FINANCIAL PII NER")
    print("=" * 70)

    # 1. Load dataset

    print("\n1. Loading Gretel synthetic financial PII dataset...")
    dataset = load_dataset("gretelai/synthetic_pii_finance_multilingual")

    train_raw = dataset["train"]
    test_raw = dataset["test"]

    print(f"   Train samples: {len(train_raw)}")
    print(f"   Test samples:  {len(test_raw)}")

    # Optional: language split
    train_en = train_raw.filter(lambda x: x["language"] == "English")
    train_fr = train_raw.filter(lambda x: x["language"] == "France")

    print(f"   English train samples: {len(train_en)}")
    print(f"   French train samples:  {len(train_fr)}")

    # 2. Initialize NER model

    print("\n2. Initializing multilingual NER model...")
    ner_model = MultilingualNERModel(model_name="xlm-roberta-base")

    # 3. Encode datasets

    print("\n3. Encoding datasets...")

    def encode_fn(example):
        return encode_finance_example(example, tokenizer=ner_model.tokenizer, label2id=ner_model.label2id)

    train_dataset = train_en.map(encode_fn, remove_columns=train_en.column_names)

    eval_dataset = test_raw.map(encode_fn, remove_columns=test_raw.column_names)

    train_dataset.set_format("torch")
    eval_dataset.set_format("torch")

    # 4. Train model

    print("\n4. Training model...")
    trainer = ner_model.train(
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        epochs=3,
        batch_size=8
    )

    print("   Training completed.")

    # 5. Evaluation

    print("\n5. Evaluating model...")
    metrics = trainer.evaluate()
    print(metrics)

    # 6. Test predictions on raw financial text

    print("\n6. Testing entity extraction...")

    test_texts = [
        "This agreement was signed on August 1, 2021 by Romeo R. Druso.",
        "The supplier is located at 406 Joshua Square and paid via IBAN DE89370400440532013000."]

    for text in test_texts:
        preds = ner_model.predict([text], language="en")[0]
        print(f"\nText: {text}")
        print("Entities:")
        for e in preds:
            print(f"  - {e['entity']}: {e['word']}")


In [ ]:
main()

### TASK B

DATASET

In [ ]:
from datasets import load_dataset

legal_dataset = load_dataset(
    "mattmdjaga/text-anonymization-benchmark-train",
    split="train[:50]"
)
finance_dataset = load_dataset(
    "gretelai/synthetic_pii_finance_multilingual",
    split="train[:50]"
)


In [ ]:
print("Legal samples:", len(legal_dataset))
print("Finance samples:", len(finance_dataset))
print(legal_dataset)
print(finance_dataset)

In [ ]:
import json
import pandas as pd
import numpy as np
from collections import Counter
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from datasets import Dataset

import json
import pandas as pd
import numpy as np
from collections import Counter
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from datasets import Dataset
import joblib
import json
import numpy as np
import pandas as pd
from collections import Counter
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from typing import List, Dict, Optional
from typing import List, Dict, Optional
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    XLMRobertaTokenizer,
    XLMRobertaForTokenClassification
)


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    XLMRobertaTokenizer, XLMRobertaForTokenClassification,
    XLMRobertaForSequenceClassification, AutoTokenizer,
    Trainer, TrainingArguments, pipeline, set_seed
)

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_recall_fscore_support
)
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import DataCollatorWithPadding


MODEL

In [ ]:
# ============================================================================
# CROSS-DOMAIN CLASSIFICATION
# ============================================================================

class DomainClassifier:
    """Classify documents into healthcare, legal, finance, or multi-domain"""

    def __init__(self, model_name="xlm-roberta-base"):
        self.model_name = model_name
        self.tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)

        # Domain labels
        self.domain_labels = [ "legal", "finance", "healthcare"]
        self.domain2id = {domain: i for i, domain in enumerate(self.domain_labels)}
        self.id2domain = {i: domain for i, domain in enumerate(self.domain_labels)}

        # Initialize model
        self.model = XLMRobertaForSequenceClassification.from_pretrained(
            model_name,
            num_labels=len(self.domain_labels)
        )

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)


    def train(self, train_texts, train_labels, val_texts=None, val_labels=None, epochs=3, batch_size=4, max_length=128):
        """
        Train the domain classifier with memory-efficient dynamic padding.

        Args:
            train_texts (List[str]): Training document texts
            train_labels (List[str]): Corresponding labels ('legal', 'finance', 'multi-domain')
            val_texts (List[str], optional): Validation texts
            val_labels (List[str], optional): Validation labels
            epochs (int): Number of epochs
            batch_size (int): Batch size (small to reduce RAM)
            max_length (int): Maximum token length
        """

        # Tokenize without padding (collator will handle dynamic padding)
        train_encodings = self.tokenizer(
            train_texts,
            truncation=True,
            padding=False,
            max_length=max_length
        )

        # Create dataset as list of dicts
        train_dataset = [
            {**{k: v[i] for k, v in train_encodings.items()}, "labels": torch.tensor(self.domain2id[train_labels[i]])}
            for i in range(len(train_texts))
        ]

        # Data collator handles padding dynamically per batch
        from transformers import DataCollatorWithPadding
        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=data_collator
        )

        # Optimizer
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=2e-5)

        # Training loop
        self.model.train()
        for epoch in range(epochs):
            total_loss = 0
            for batch in train_loader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                optimizer.zero_grad()
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                loss = outputs.loss
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

        # Validation if provided
        if val_texts and val_labels:
            accuracy = self.evaluate(val_texts, val_labels)
            print(f"Validation Accuracy: {accuracy:.4f}")

    def predict(self, texts: List[str]) -> List[Dict]:
        """Predict domain for texts"""

        self.model.eval()
        encodings = self.tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=256,
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**encodings)
            predictions = torch.argmax(outputs.logits, dim=1)

        results = []
        for text, pred in zip(texts, predictions):
            domain = self.id2domain[pred.item()]

            # softmax over classes
            probs = torch.nn.functional.softmax(outputs.logits, dim=1)
            confidence = probs[0, pred.item()].item()  # batch index 0, class index pred

            # Detect if it might be multi-domain based on keywords
            keywords_found = self.detect_keywords(text)

            results.append({
                "text": text[:100] + "..." if len(text) > 100 else text,
                "predicted_domain": domain,
                "confidence": confidence,
                "keywords_detected": keywords_found,
                "suggested_domain": self.adjust_domain(domain, keywords_found)
            })

        return results

    def detect_keywords(self, text: str) -> Dict[str, List[str]]:
        """Detect domain-specific keywords in text"""

        keywords_found = {domain: [] for domain in [ "legal", "finance","healthcare"]}
        text_lower = text.lower()

        for domain, keywords in {
            "healthcare": ["patient", "diagnosis", "treatment"],
            "legal": ["contract", "agreement", "clause", "liability", "jurisdiction"],
            "finance": ["investment", "transaction", "portfolio", "dividend", "valuation"]
        }.items():
            for keyword in keywords:
                if keyword in text_lower:
                    keywords_found[domain].append(keyword)

        return keywords_found

    def adjust_domain(self, predicted_domain: str, keywords_found: Dict) -> str:
        """Adjust domain prediction based on keyword detection"""

        # Count keywords per domain
        domain_counts = {domain: len(keywords) for domain, keywords in keywords_found.items()}

        # If multiple domains have significant keyword counts, suggest multi
        significant_domains = [domain for domain, count in domain_counts.items()
                              if count >= 2]

        if len(significant_domains) > 1:
            return "multi-domain"
        elif len(significant_domains) == 1:
            return significant_domains[0]
        else:
            return predicted_domain

    def evaluate(self, texts: List[str], true_labels: List[str]) -> float:
        """Evaluate classifier performance"""

        predictions = self.predict(texts)
        pred_labels = [p["predicted_domain"] for p in predictions]

        # Calculate accuracy
        accuracy = accuracy_score(true_labels, pred_labels)

        # Confusion matrix
        cm = confusion_matrix(true_labels, pred_labels, labels=self.domain_labels)

        # Plot confusion matrix
        self.plot_confusion_matrix(cm)

        # Classification report
        print("\nClassification Report:")
        print(classification_report(true_labels, pred_labels, target_names=self.domain_labels))

        return accuracy

    def plot_confusion_matrix(self, cm):
        """Plot confusion matrix"""

        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                   xticklabels=self.domain_labels,
                   yticklabels=self.domain_labels)
        plt.title("Domain Classification Confusion Matrix")
        plt.ylabel("True Domain")
        plt.xlabel("Predicted Domain")
        plt.tight_layout()
        plt.savefig("domain_confusion_matrix.png", dpi=150)
        plt.show()




In [ ]:
# loading 3 datasets
legal_dataset = load_dataset(
    "mattmdjaga/text-anonymization-benchmark-train",
    split="train[:50]"
)
finance_dataset = load_dataset(
    "gretelai/synthetic_pii_finance_multilingual",
    split="train[:50]"
)
health_dataset = load_dataset(
    "TypicaAI/MedicalNER_Fr",
    split="train[:50]"
)
print("\nSTEP 2: Loaded datasets")
print(f"Legal: {len(legal_dataset)} samples")
print(f"Finance: {len(finance_dataset)} samples")
print(f"health: {len(health_dataset)} samples")


# STEP 1 PROCESS LEGAL DATASET
import json

def process_legal_document(doc, index):
    text = doc.get("text", "")
    entities = []

    annotations = doc.get("annotations")

    # annotations must be a dict
    if not isinstance(annotations, dict):
        annotations = {}

    # Loop over annotators (e.g., annotator1)
    for annotator, data in annotations.items():
        #  FIX: data may be None
        if not isinstance(data, dict):
            continue

        mentions = data.get("entity_mentions", [])
        if not isinstance(mentions, list):
            continue

        for m in mentions:
            start = m.get("start_offset")
            end = m.get("end_offset")

            if start is None or end is None:
                continue

            entities.append({
                "start": start,
                "end": end,
                "label": m.get("entity_type", "UNKNOWN"),
                "text": text[start:end]
            })

    return {
        "id": doc.get("doc_id", f"legal_{index}"),
        "text": text,
        "entities": entities,
        "domain": "legal",
        "language": "en"
    }



legal_data = [process_legal_document(doc, i) for i, doc in enumerate(legal_dataset)]
print(f"Processed {len(legal_data)} legal documents")

# STEP 2: PROCESS FINANCE DATASET
import json

def process_finance_document(doc, index):
    text = doc.get("generated_text", "")
    entities = []

    pii_spans = doc.get("pii_spans", [])

    # Parse JSON string safely
    if isinstance(pii_spans, str):
        try:
            pii_spans = json.loads(pii_spans)
        except Exception:
            pii_spans = []

    if isinstance(pii_spans, list):
        for span in pii_spans:
            start = span.get("start")
            end = span.get("end")

            if start is None or end is None:
                continue

            entities.append({
                "start": start,
                "end": end,
                "label": span.get("label", "UNKNOWN").upper(),
                "text": text[start:end]
            })

    return {
        "id": str(doc.get("index", f"finance_{index}")),
        "text": text,
        "entities": entities,
        "domain": "finance",
        "language": "en" if doc.get("language", "").lower().startswith("english") else doc.get("language", "en"),
        "metadata": {
            "document_type": doc.get("document_type", "Unknown"),
            "quality_score": doc.get("quality_score", None),
            "conformance_score": doc.get("conformance_score", None)
        }
    }


finance_data = [process_finance_document(doc, i) for i, doc in enumerate(finance_dataset)]
print(f" Processed {len(finance_data)} finance documents")

# STEP 3: PROCESS HEALTHCARE DATASET

def process_healthcare_document(doc, index):
    text = doc.get("text", "") or " ".join(doc.get("tokens", []))
    entities = []

    # TypicaAI/MedicalNER_Fr provides ner_tags_span field
    spans = doc.get("ner_tags_span", [])
    if isinstance(spans, str):
        try:
            spans = json.loads(spans)
        except:
            spans = []

    for s in spans:
        if len(s) != 3:
            continue
        label, start, end = s
        entities.append({
            "start": start,
            "end": end,
            "label": label.upper(),
            "text": text[start:end]
        })

    return {
        "id": doc.get("id", f"health_{index}"),
        "text": text,
        "entities": entities,
        "domain": "healthcare",
        "language": "fr"
    }

health_data = [process_healthcare_document(doc, i) for i, doc in enumerate(health_dataset)]
print(f" Processed {len(health_data)} healthcare documents")

# STEP 4: COMBINE ALL DOCUMENTS

all_documents = legal_data + finance_data + health_data
all_labels = [0]*len(legal_data) + [1]*len(finance_data) + [2]*len(health_data)
print(f" Total documents: {len(all_documents)}")

# STEP 5: SPLIT DATA train val

from sklearn.model_selection import train_test_split

train_texts = [doc["text"] for doc in all_documents]
train_labels = [doc["domain"] for doc in all_documents]

X_train, X_val, y_train, y_val = train_test_split(
    train_texts,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels
)
print(f"Training samples: {len(X_train)}, Validation samples: {len(X_val)}")



Trainning

In [ ]:
classifier = (model_name="xlm-roberta-base")

classifier.train(
    train_texts=X_train,
    train_labels=y_train,
    val_texts=X_val,
    val_labels=y_val,DomainClassifier
    epochs=3
)


In [ ]:
sample_texts = [
    legal_data[0]["text"],
    finance_data[0]["text"],
    health_data[0]["text"]
]

predictions = classifier.predict(sample_texts)


In [ ]:
final_results = []

for doc, pred in zip(
    [legal_data[0], finance_data[0], health_data[0]],
    predictions
):
    final_results.append({
        "document_id": doc["id"],
        "true_domain": doc["domain"],
        "predicted_domain": pred["predicted_domain"],
        "confidence": pred["confidence"],
        "entities": doc["entities"]
    })

import json
print(json.dumps(final_results, indent=2))
